This part of the pipeline processes the raw antiSMASH output and statistically compares the normalised BGC counts by order.

### Paths and parameters

#### Libraries

In [ ]:
from Bio import SeqIO
import os
from os.path import join
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools as it
import scipy.stats as sts
import numpy as np
from statannotations.Annotator import Annotator

#### Pipeline input folders

In [ ]:
metadata_file = "dereplicated_metadata"
classification_file = "02-GTDB/filtered_classification_table"

#### Pipeline output folders

In [ ]:
task_root = "09-MGEs/BGCs"
output_folder = join(task_root, "output")
results_folder = join(task_root, "processed_output")

#### Tool pointers and parameters

#### Other setups

In [ ]:
custom_palette = sns.husl_palette()
custom_palette = [custom_palette[0], custom_palette[1], custom_palette[2], custom_palette[4]]
custom_palette

In [ ]:
os.makedirs(results_folder, exist_ok=True)

## Reading input files

### Parsing the output

This parser will assign all hits to a certain BGC type so that we can differentiate these downstream, but this causes hybrid regions to be included multiple times, so we'll have to deduplicate these in the general count plots later on.

In [ ]:
os.makedirs(results_folder, exist_ok=True)
result_dirs = os.listdir(output_folder)
hits_hybrids = []
# Take a look into the result directory of each screened assembly
for dir in result_dirs:
    dir_conts = os.listdir(join(output_folder, dir))
    genbank_files = [f for f in dir_conts if '.region' in f]
    # Take a look into each GenBank region file
    for gbf in genbank_files:
        with open(join(output_folder, dir, gbf), "r") as handle:
            seq = list(SeqIO.parse(handle, 'genbank'))[0]
        # Get all product region tags in this file
        region_tags = [f for f in seq.features if f.type == "region"][0]
        types = region_tags.qualifiers['product']
        hybrid = len(types) != 1
        # Create a hit record for each BGC type in this region
        for type in types:
            length = int(region_tags.location.end)
            contig = seq.id
            region = int(region_tags.qualifiers['region_number'][0])
            record = {'assembly_ID': dir, 'contig_ID': contig, 'region': region, 'type': type, 'hybrid': hybrid, 'length': length}
            hits_hybrids.append(record)
hits_hybrids = pd.DataFrame(hits_hybrids).set_index('assembly_ID')
hits_hybrids

In [ ]:
hits_hybrids.to_csv(join(results_folder, "all_hits"), sep = '\t', index = False)

### GTDB orders

In [ ]:
classification = pd.read_table(classification_file, sep = "\t", header = None, names = ['accession', 'group'])
classification = classification.set_index('accession').squeeze()
classification

In [ ]:
plot_order = sorted(list(classification.unique()))

### Reading the genome sizes

Necessary for normalising the general BGC counts

In [ ]:
sizes = pd.read_table(metadata_file, usecols = [0,4], sep = "\t")
sizes = sizes.rename(columns = {'Assembly Stats Total Sequence Length': 'size',
                               'Assembly Accession': 'accession'})
sizes = sizes.set_index('accession').squeeze()
sizes

### Adding metadata

In [ ]:
hits_hybrids = pd.merge(hits_hybrids, classification, left_index = True, right_index = True)
hits_hybrids = pd.merge(hits_hybrids, sizes, left_index = True, right_index = True)
hits_hybrids

### General count plots

Let's now deduplicate the records of the hybrid regions after omitting the `hybrid` and `type` columns.

In [ ]:
hits = hits_hybrids.drop(columns = ['hybrid', 'type']).drop_duplicates()

Add metadata columns and normalise the general counts by genome size

In [ ]:
cluster_counts = pd.DataFrame(hits.groupby('assembly_ID')['contig_ID'].count()
                             ).reset_index().rename(columns = {'contig_ID': 'No. BGCs'})
cluster_counts['group'] = cluster_counts['assembly_ID'].apply(lambda x: classification[x])
cluster_counts['size'] = cluster_counts['assembly_ID'].apply(lambda x: sizes[x])
cluster_counts['Norm. no. BGCs'] = cluster_counts['No. BGCs']/cluster_counts['size']*1000000
cluster_counts

#### Barplot

In [ ]:
fig, ax = plt.subplots(figsize = (5,2))
ax = sns.barplot(ax = ax, data = cluster_counts, estimator = "mean", errorbar = "se",
                 x = "Norm. no. BGCs", y = "group", palette = custom_palette,
                 width = 0.9, orient = "h", order = plot_order)
plt.xlabel('Avg. norm. no. BGCs')
plt.ylabel('GTDB order')
plt.title('BGCs')
plt.savefig(join(results_folder, "av_counts_BGCcluster_bar.svg"))
plt.show()

#### Violinplot

In [ ]:
fig, ax = plt.subplots(figsize = (5,3))
ax = sns.violinplot(ax = ax, data = cluster_counts, x = 'Norm. no. BGCs', y = 'group',
                    palette = custom_palette, orient = 'h', cut = 0, order = plot_order)
plt.xlabel('Norm. no. BGCs')
plt.ylabel('GTDB order')
plt.title('BGCs')

# Adding statistical significance marks
pairs = list(it.combinations(cluster_counts['group'].unique(), 2))
annotator = Annotator(ax = ax, pairs = pairs, data = cluster_counts, x = 'Norm. no. BGCs', y = 'group', orient = 'h', cut = 0, 
                      order = plot_order)
annotator.configure(test = 'Brunner-Munzel', comparisons_correction="Bonferroni", text_format = 'star', loc = 'inside')
annotator.apply_and_annotate()

plt.savefig(join(results_folder, 'counts_BGCcluster_violin.svg'))
plt.show()

#### Exact stats

Getting all counts grouped by order

In [ ]:
cluster_counts_stats = cluster_counts[['Norm. no. BGCs', 'group']].to_dict(orient = 'list')
cluster_counts_stats = list(zip(*cluster_counts_stats.values()))
counts_stats = {}
for record in cluster_counts_stats:
    try:
        counts_stats[record[1]].append(record[0])
    except KeyError:
        counts_stats[record[1]] = [record[0]]
counts_stats

In [ ]:
[(i, [np.mean(j), np.std(j)]) for i,j in counts_stats.items()]

In [ ]:
tests = list(it.combinations(counts_stats.keys(), 2)) # get all order pairs
for comb in tests:
    p_val = sts.brunnermunzel(counts_stats[comb[0]], counts_stats[comb[1]])[1]
    q_val = min(p_val * len(tests), 1) # Bonferroni correction
    print(f'{comb}: {q_val}')

### BGC types per cluster

Count by BGC type. Hybrids count for each region they're a hybrid of.

In [ ]:
type_counts = pd.DataFrame(hits_hybrids.groupby(['assembly_ID', 'type'])['contig_ID'].count()
                               ).reset_index().rename(columns = {'contig_ID': 'No. BGCs'})
type_counts

Readd order annotation and melt the dataframe by cluster annotation

In [ ]:
type_counts_pivot = type_counts.pivot(columns = "type", index = "assembly_ID", values = "No. BGCs").fillna(0).astype(int)
type_counts_pivot['group'] = type_counts_pivot.index.to_series().apply(lambda x: classification[x])
type_counts_pivot = type_counts_pivot.melt(id_vars = 'group').rename(columns = {'value': 'No. BGCs'})
type_counts_pivot

In [ ]:
fig, ax = plt.subplots(figsize = (6,8))
ax = sns.barplot(ax = ax, data = type_counts_pivot, estimator = "mean", errorbar = "se",
                 x = "No. BGCs", y = "type", hue = "group", order = sorted(type_counts['type'].unique()),
                 palette = custom_palette, width = 0.9, orient = "h")
plt.xlabel('Avg. no. BGCs')
plt.ylabel('BGC class')
plt.savefig(join(results_folder, "av_counts_BGCtype.svg"))
plt.show()